# **Proyecto final - Desastres naturales y su impacto en la población de méxico**
**Auto:** Marian Alejandra Herrera Ayala  **| Ultima Actualizacion:** *Julio - 2026*

# **Análisis del proyecto**

---

En el norte del país se vive la peor sequía vista en décadas, en estados como Coahuila, Chihuahua, Nuevo León, Sinaloa, Sonora y Tamaulipas presentan municipios que presentan cierto grado de sequía.

En el sur de Sonora, son ocho los municipios de afectados por sequía registrada y otros 16 están anormalmente secos, según la CONAGUA.

Presas como Álvaro Obregón, o más conocida como el Oviáchic, es la principal fuente de agua para los productores del Valle del Yaqui, tan solo reportó un nivel de agua del 16.7% de su capacidad.

Según el reporte más reciente al mes de abril del presente año del Distrito de Riego del Río Yaqui, el embalse solo cuenta con 539.3 millones de metros cúbicos de agua, que si bien presenta una mejoría con respecto a la del año pasado, no es suficiente para proporcionar estabilidad y confianza para continuar con las labores del sector agrícola cómodamente.

Por otro lado, en estados en el centro y sur del país es más común ver eventos como tormentas, huracanes, lluvias intensas, etc. La gran capacidad de obtener agua en esos estados es con creces sumamente superior a la de los estados norteños, donde ver lluvias constantes, incluso en temporada de lluvía, es algo sumamente inusual.

En estados donde la temperatura natural se define entre los 38 °C y los 45 °C, y solo contamos con 2 estaciones al año casi perpetuas de Verano y un mes de invierno, y retornar a verano, no se puede evitar pensar que la región sufre de una predilección histórica de sufrir por problemas de abastecimiento de agua por la sequía y la poca cantidad de lluvias tanto regulares como atípicas.

Mientras que la región central y sur del país tiene un fuerte y consistente historial de tormentas, lluvias e inundaciones, que se les puede considerar atípicas. Aunque también se les ha visto propensos a eventos geológicos como sismos y terremotos poco frecuentes en el norte, también una realidad distinta para analizar.


**Objetivo**
---

Dado a entender esté contexto, la finalidad de esté proyecto es tratar de darle sentido a los distintos desastres que han ocurrido en México a lo largo de los años y tratar de identificar si existe algún patrón en cuanto a la tendencia de los desastres naturales reportados en las distintas regiones del país.

**Preguntas base del proyecto**
---

* Verdaderamente, ¿en mi natal estado de Sonora ha empeorado la sequía a lo largo de los últimos 20 años? De ser así, responder a la pregunta, ¿Cuánto ha empeorado la sequía en mi comunidad a lo largo de los años? O en caso contrario, ¿cuanto a disminuido está problematica?
* ¿En qué estados hay más desastres naturales y cuales son el tipo más frecuente y en que grado?
* ¿Cuáles son los mayores desastres naturales a nivel región?
* ¿Qué incidentes con muchas defunciones o daños altos no tuvieron declaratoria oficial y/o contaron con una información temporal mal definida?
* ¿Cuáles son los eventos que tienen un mayor impacto económico y con qué frecuencia pasan?



**Hipotesis**
---

En este punto, considero que los mayores fenómenos naturales ocurridos en el país están relacionados con el agua, especialmente en la zona sur y centro del país, mientras que el norte se experimentan más problemas relacionados con la temperatura y la sequía, así como lluvias ocasionales.

Sumado a ello, la mayor perdida economica se debe de concentrar en el sur del país, gracias a las distintas tormentas trópicales y hurácanes que azotan la región.

# **Importaciones**

---

In [918]:
import os
import pandas as pd
import re
import unicodedata
import requests
import time
from google.colab import userdata
from functools import reduce
import matplotlib.pyplot as plt

In [919]:
import subprocess
import sys

try:
    import pymannkendall as mk
except ImportError:
    print("pymannkendall no está instalado, instalando...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pymannkendall', '-q'])
    import pymannkendall as mk

print("Listo, pymannkendall disponible")

Listo, pymannkendall disponible


In [920]:
ruta = 'https://github.com/saalej/Desastres_Naturales_MX_2000_2024/raw/refs/heads/main/Desastres/'

try:
    TOKEN_INEGI = userdata.get('TOKEN_INEGI')
    TOKEN_INEGI_2 = userdata.get('TOKEN_INEGI_2')
except Exception:
    print("⚠️ TOKEN_INEGI no configurado en Secrets — se usarán los CSV ya guardados.")
    TOKEN_INEGI = None
    TOKEN_INEGI_2 = None

# **Funciones auxiliares**

---

In [921]:
estados_por_region = {
    'Norte': [
        'Baja California',
        'Baja California Sur',
        'Sonora',
        'Chihuahua',
        'Coahuila de Zaragoza',
        'Nuevo León',
        'Tamaulipas',
        'Sinaloa',
        'Durango'],
    'Occidente': [
        'Jalisco',
        'Colima',
        'Nayarit',
        'Michoacán de Ocampo',
        'Guanajuato',
        'Zacatecas',
        'Aguascalientes',
        'San Luis Potosí'],
    'Centro': [
        'Ciudad de México',
        'México',
        'Hidalgo',
        'Tlaxcala',
        'Puebla',
        'Morelos',
        'Querétaro'],
    'Sur': [
        'Guerrero',
        'Oaxaca',
        'Chiapas'],
    'Sureste': [
        'Veracruz de Ignacio de la Llave',
        'Tabasco',
        'Campeche',
        'Yucatán',
        'Quintana Roo']
}

In [922]:
# Línea de separación
def printSeparador(largo = 30):
  print('-' * largo)

**Mostrar información general de un DataFrame**

1. `.shape` - Obtener el tamaño del dataset (Registros, columnas) para darme una idea de la cantidad de información que manejare.
2. `.info` - Obtener un resumen general del dataset, incluyendo el tipo de la columna que eventualmente usarse para determinar si debo modificar los tipos de dato utilizados y determinar existencia de nulos y casos NaN.

In [923]:
def showDataFrame(data):
  print(data.shape)
  printSeparador()
  print(data.info())
  printSeparador()
  print(f"Registros duplicados\n{data.isnull().sum()}")

**Elimina espacios al principio y al final**

Elimina espacios en blanco al inicio y final de TODAS las columnas de tipo texto (object) en un DataFrame. No toca columnas numéricas, de fecha, ni booleanas -- solo texto.

In [924]:
def limpiar_espacios(df):
    df = df.copy()

    # 1. Limpiar espacios en los NOMBRES de columna
    df.columns = df.columns.str.strip()

    # 2. Limpiar espacios en los VALORES de cada columna de texto
    columnas_texto = df.select_dtypes(include='object').columns
    for col in columnas_texto:
      # .str.strip() ignora automáticamente los NaN, no los rompe
      df[col] = df[col].str.strip()

    return df

**Normalizacion del texto en las celdas**

Convierte el texto de las columnas indicadas a formato: Primera letra mayúscula, resto minúsculas.

Si el primer carácter es número o símbolo, lo deja igual (no tiene mayúscula que aplicar).

In [925]:
def normalizar_capitalizacion(df, columnas=None):
    df = df.copy()

    if columnas is None:
        columnas = df.select_dtypes(include='object').columns

    for col in columnas:
        df[col] = df[col].str.capitalize()

    return df

**Normalización de columnas**

Los nombres originales de las columnas los paso a una nomenclatura camal case. Por lo tanto, eliminando los espacios entre palabras, acentos, poniendo todo a minúsculas, así cómo quitar guiones bajos innecesarios. Así mismo se hizo una distinción especial por la palabra año, que al convertirse directamente genera una palabra que puede resultar un tanto ofensiva, así que se agregó una corrección personalizada para está evitar incidencias.

In [926]:
def normalizacion_columnas(data_frame):
  nuevas_columnas = {}

  # Normalizacion de las columnas
  for columna in data_frame.columns.tolist():
      # Quitar acentos
      nombre = unicodedata.normalize('NFKD', columna).encode('ascii', 'ignore').decode('utf-8')

      # Quitar paréntesis y caracteres especiales (dejar solo letras, numeros y espacios)
      nombre = nombre.lower()

      # Reemplazar uno o más espacios por un solo guion bajo
      nombre = re.sub(r'\s+', '_', nombre)

      # Caso especial: Año > anio
      nombre = re.sub(r'(?<![a-z])ano(?![a-z])', 'anio', nombre)

      # Quitar guiones bajos sobrantes al inicio o final (por si quedó alguno)
      nombre = nombre.strip('_')

      nuevas_columnas[columna] = nombre

  data_frame = data_frame.rename(columns=nuevas_columnas)
  return data_frame

# **Sección 1. Importanto datasets**

---

*Listado de dataset a utilizar en este proyecto*

* Base de datos sobre el impacto socioeconómico de los daños y pérdidas ocasionados por los desastres en México de 2000 a 2024
* Sistema de Consulta de Declaratorias 2000-2023
* Censo poblacional INEGI 2020
* Área geográfica - Estados Unidos Mexicanos INEGI




**Base de datos sobre el impacto socioeconómico de los daños y pérdidas ocasionados por los desastres en México de 2000 a 2024**
---


Este dataset es un csv descriptivo sobre desastres naturales de México documentales históricamente desde el año 2000 hasta 2024. El documento fue obtenido del Atlas Nacional de Riesgos, que declara fue elaborado por el  la Subdirección de Estudios Económicos y Sociales de la Dirección de Análisis y Gestión de Riesgo,

Este documento es el resultado de un análisis exhaustivo y de la posterior unificación de las diversas fuentes, tanto públicas y privadas, consultadas, así como también de investigaciones de campo y consultas directas con autoridades locales.

Fue obtenido de la página oficinal del organismo gubernamental  `Centro Nacional de Prevención de Desastres (CENAPRED)`, en su apartado de `Descargas` en su sección `Evaluación del Impacto Económico y Social de los Desastres en México`, bajo el nombre mencionado como título en está sección del proyecto.

El propósito de utilizar este Dataset es utilizarlo como la base de mi dataset final, gracias a que cuenta con información cuantificable referentes a los desastres tales como:

1. Total de defunciones
2. Población afectada
3. Viviendas dañadas
4. Escuelas afectadas
5. Hospitales afectados
6. Comercios afectados
7. Área de cultivo dañada / pastizales (h)
8. Total de daños (millones de pesos)

Así mismo cuenta con información cualificable como fechas, ubicación y en la mayoría de los casos una descripción general sobre lo ocurrido.

---


La importación de este dataset será directamente desde mis archivos de Drive, por lo que utilizare un archivo csv para realizar la importación utilizando una codificación que puede interpretar correctamente los acentos *(á, é, í, ó, ú), la ñ, y símbolos como €, ¿, ¡*.


Esto se hace pensando que el csv cuenta con información geográfica del país, los cuales es común encontrar acentos, y evitaremos confusiones con nombres que no son los oficiales para dichas comunidades.


Posteriormente, se hace uso de la función de *pandas* `read_csv` para leer el csv original y convertirlo a un DataFrame analizable y manipulable por mí para cumplir los fines de este proyecto.


In [927]:
base_historica = 'Basehistorica_2000_a_2024.csv'

archivo_unificado = ruta + base_historica

df_base_historica = pd.read_csv(archivo_unificado, encoding='latin1')
df_base_historica = limpiar_espacios(df_base_historica)
df_base_historica = normalizacion_columnas(df_base_historica)

showDataFrame(df_base_historica)

(10354, 17)
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10354 entries, 0 to 10353
Data columns (total 17 columns):
 #   Column                                   Non-Null Count  Dtype 
---  ------                                   --------------  ----- 
 0   fecha_de_inicio                          10354 non-null  object
 1   fecha_de_fin                             10351 non-null  object
 2   anio                                     10354 non-null  int64 
 3   clasificacion_del_fenomeno               10354 non-null  object
 4   tipo_de_fenomeno                         10354 non-null  object
 5   estado                                   10354 non-null  object
 6   municipios_afectados                     10322 non-null  object
 7   descripcion_general_de_los_danos         10353 non-null  object
 8   defunciones                              10354 non-null  object
 9   poblacion_afectada                       10354 non-null  object
 10  viviendas_danad

In [928]:
df_base_historica.sample(10)

,fecha_de_inicio,fecha_de_fin,anio,clasificacion_del_fenomeno,tipo_de_fenomeno,estado,municipios_afectados,descripcion_general_de_los_danos,defunciones,poblacion_afectada,viviendas_danadas,escuelas,hospitales,comercios,area_de_cultivo_danada_/_pastizales_(h),total_de_danos_(millones_de_pesos),fuente
748,8-mar,8-mar,2003,Sociorganizativo,Accidente de transporte,México,SD,Se registró el desplome de una avioneta de la ...,3,4,0,0,0,0,0,$1.00,CENACOM
9501,21-sep,21-sep,2022,Sociorganizativo,Accidente de transporte,Guanajuato,Abasolo,Se registró choque con volcadura entre camión ...,0,21,0,0,0,0,0,$0.03,CENACOM
2436,11-ene,11-ene,2007,Sociorganizativo,Accidente de transporte,Jalisco,San Juan de los Lagos,Volcadura de una pipa que transportaba acido m...,0,0,0,0,0,0,0.00,$0.27,CENACOM
2788,6-abr,6-abr,2008,Químico,Incendio urbano,México,Ecatepec,Se registró un incendio en un basurero quemand...,0,0,0,0,0,0,0.00,SD,CENACOM
5231,13-jun,13-jun,2013,Hidrometeorológico,Lluvias,San Luis Potosí,Villa de Ramos y Tamasopo,Lluvias fuertes que ocasionaron en los municip...,0,87,20,0,0,0,0,$0.02,CENACOM
6003,6-feb,6-feb,2015,Sociorganizativo,Accidente de transporte,Guanajuato,Silao,"Carambola entre ocho vehículos, entre ellos un...",0,18,0,0,0,0,0,$0.42,CENACOM
255,11-abr,11-abr,2002,Químico,Incendio forestal,México,SD,"Del total presentado, según autoridades del go...",0,0,0,0,0,0,0,$88.10,CENACOM
7047,31-oct,31-oct,2018,Químico,Fuga,Michoacán,Cuitzeo,"A las 11:36 horas del 31 de octubre, se report...",0,0,0,0,0,0,0,$0.51,CENACOM
303,10-jun,10-jun,2002,Sociorganizativo,Concentración masiva de población,Guerrero,SD,Los comerciantes de este centro turístico sufr...,0,0,0,0,0,0,0,$12.00,CENACOM
929,12-jul,12-jul,2004,Hidrometeorológico,Lluvias,Sonora,Arivechi.,Se reportaron fuertes lluvias en el municipio ...,0,165,33,0,0,0,300,$4.70,CENACOM


En base a la visualización de la información de manera aleatoria, así como de los tipos de datos que actualmente se usa en el data frame, puede llegar a las siguientes observaciones:

**Correcciones**
1.   Reemplazar $- por un valor (float) 0.00 en columa total_de_danos(millones_de_pesos), eventualmente será lo mejor para hacer cálculos más tarde
2.   Convertir los datos a un tipo numérico en el resto de columnas que usan datos cuantitativos
3. Descartar la columna anio y usar el dato para completar la `fecha_de_inicio` y la `fecha_de_fin`. Posteriormente, convertir los datos de correspondientes a las fechas a un formato para adecuado (***YYYY-mm-DD***)
5. Descartar registros cuya clasificación del fenómeno no corresponde a un desastre natural (Hidrometeorológico, Geológico, etc.) [Revisar la información disponible para escoger las clasificaciones necesarias]
6. Descartar columna `Fuente` no es relevante para el estudio


**Revisión de casos**
1. Revisar cómo manejar desastres como incendio que entra en la clasificación del fenómeno químico, pero pueden ser tanto naturales como artificiales.
2. En el caso de la columna `estado` y `municipios_afectados` habrá que revisar cómo se empalma con los demás dataset al combinarse y eventualmente definir como filtraremos y usaremos está información.
3. Revisar registros nulos

**Sistema de Consulta de Declaratorias 2000-2023**
---

Esté dataset lo usaré para terminar de complementar los municipio afectados por ciertos desastres naturales, en el dataset **`df_base_historica`** a pesar de ser más completo estadísticamente hablando no me proporciona el tipo de declaratoria con el que fue considerado al reportarse el incidente, así que quiero utilizar esté dataset para darle una clasificación de

In [929]:
atlas = 'atlas_nacional_riesgo_completo.csv'

archivo_unificado = ruta + atlas

df_atlas = pd.read_csv(archivo_unificado, encoding='latin1')
df_atlas = limpiar_espacios(df_atlas)
df_atlas = normalizacion_columnas(df_atlas)

showDataFrame(df_atlas)

(34453, 12)
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34453 entries, 0 to 34452
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   identificador           34453 non-null  int64 
 1   estado                  34453 non-null  object
 2   clave_estado            34453 non-null  int64 
 3   municipio               34453 non-null  object
 4   clave_municipio         34453 non-null  int64 
 5   tipo_declaratoria       34453 non-null  object
 6   clasificacion_fenomeno  34453 non-null  object
 7   tipo_fenomeno           34453 non-null  object
 8   fecha_publicacion       34453 non-null  object
 9   fecha_inicio            34453 non-null  object
 10  fecha_fin               34453 non-null  object
 11  observaciones           34453 non-null  object
dtypes: int64(3), object(9)
memory usage: 3.2+ MB
None
------------------------------
Registros duplicados
identificador

In [930]:
df_atlas.sample(10)

,identificador,estado,clave_estado,municipio,clave_municipio,tipo_declaratoria,clasificacion_fenomeno,tipo_fenomeno,fecha_publicacion,fecha_inicio,fecha_fin,observaciones
21687,26091,Oaxaca,20,Asunción Cuyotepeji,20004,Emergencia,Geológico,Sismo,02/10/2017,19/09/2017,19/09/2017,Sismo magnitud 7.1 Puebla-Morelos.\nEl municip...
17785,21228,Veracruz de Ignacio de la Llave,30,Altotonga,30010,Emergencia,Hidrometeorológico,Lluvias,30/10/2015,16/10/2015,20/10/2015,Lluvia severa. Declaratoria emitida el 30 de O...
16624,19951,Sonora,26,Naco,26039,Desastre,Hidrometeorológico,Ciclón Tropical,25/09/2014,16/09/2014,17/09/2014,"T. T. ""Odile"""
33056,26988,Yucatán,31,Cenotillo,31012,Desastre,Hidrometeorológico,Ciclón Tropical,26/10/2020,01/10/2020,08/10/2020,Huracán Delta
12950,14976,Veracruz de Ignacio de la Llave,30,Tuxtilla,30190,Contingencia Climatológica,Hidrometeorológico,Sequía,07/10/2011,01/01/2011,31/03/2011,Desastre Sagarpa
25970,26379,San Luis Potosí,24,Armadillo de los Infante,24004,Desastre,Hidrometeorológico,Sequía,11/04/2001,01/05/2000,31/12/2000,Sequía atípica y prolongada de mayo a diciembre.
19979,25161,Chihuahua,8,Galeana,8023,Emergencia,Hidrometeorológico,"Nevadas, Heladas, Granizadas",21/12/2001,18/12/2001,18/12/2001,"Heladas, Nevadas y Bajas Temperaturas"
27185,26424,Oaxaca,20,Santa Ana Tavela,20357,Emergencia,Químico,Incendio Forestal,22/04/2003,01/01/2003,27/03/2003,Incendio forestal. En la declaratoria de desas...
10913,12110,Sonora,26,San Ignacio Río Muerto,26072,Contingencia Climatológica,Hidrometeorológico,Ciclón Tropical,31/12/2009,03/09/2009,03/09/2009,Sin Observaciones
29647,26681,Chiapas,7,El Porvenir,7070,Desastre,Geológico,Sismo,14/02/2019,01/02/2019,01/02/2019,Sismo magnitud 6.5


**Catalogo de estados y área geográfica (INEGI)**
---

In [931]:
poblacion = 'poblacion_inegi_por_estado.csv'
catalogo_estados = 'catalogo_estados_inegi.csv'

In [932]:
def get_url_poblacional(clave, token):
  return f'https://inegi.org.mx/app/api/indicadores/desarrolladores/jsonxml/INDICATOR/1002000001/es/{str(clave).zfill(2)}/false/BISE/2.0/{token}?type=json'

# Funcion para obtener los datos poblacionales historicos por estado (CENSO)
def obtener_datos_inegi(clave_estado, token):
  url = get_url_poblacional(clave_estado, token)

  try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    datos  = response.json()
    observaciones = datos['Series'][0]['OBSERVATIONS']

    resultados_estado = []
    for obs in observaciones:
      if obs['OBS_VALUE'] is None:
        continue;
      resultados_estado.append({
        'clave_estado': clave_estado,
        'anio_poblacion': int(obs['TIME_PERIOD']),
        'poblacion': float(obs['OBS_VALUE'])
      })
    return resultados_estado
  except Exception as e:
    print(f"Error con clave {clave_estado}: {e}")
    return []

In [933]:
urlCatalogoAreasGeograficas = f'https://www.inegi.org.mx/app/api/indicadores/desarrolladores/jsonxml/CL_GEO_AREA/0/es/BISE/2.0/{TOKEN_INEGI_2}?type=json'

def  generar_catalogo_areas_geograficas():
  try:
    response = requests.get(urlCatalogoAreasGeograficas, timeout=10)
    response.raise_for_status()
    datos  = response.json()

    df_catalogo = pd.DataFrame(datos['CODE'])
    df_catalogo.columns = ['clave_estado', 'estado']

    # Elimino datos no relevantes: 00, 33-39
    # No lo convierto a int clave_estado porque no es un valor que usare de manera
    # cuantitativa, sino uno que se usa como valor cualitativo
    df_catalogo = df_catalogo[
        df_catalogo['clave_estado'].str.match(r'^\d{2}$') &
        df_catalogo['clave_estado'].astype(int).between(1, 32)
    ]

    df_catalogo = df_catalogo.reset_index(drop=True)
    df_catalogo.to_csv(ruta + catalogo_estados, index=False)
    return df_catalogo
  except Exception as e:
    print(f"Error Catalogo Geografico: {e}")

def generar_archivo_poblacional(catalogo):
  resultados = []
  for clave in catalogo['clave_estado']:
      resultado = obtener_datos_inegi(clave, TOKEN_INEGI)
      resultados.extend(resultado)
      time.sleep(0.5)

  df_poblacion = pd.DataFrame(resultados)
  df_poblacion = df_poblacion.sort_values(by=['clave_estado', 'anio_poblacion'])
  df_poblacion = df_poblacion.reset_index(drop=True)

  df_poblacion.to_csv(ruta + poblacion, index=False)
  return df_poblacion


In [934]:
df_catalogo = pd.read_csv(ruta + 'catalogo_estados_inegi.csv', dtype={'clave_estado': str})
df_poblacion = pd.read_csv(ruta + 'poblacion_inegi_por_estado.csv', dtype={'clave_estado': str})

df_catalogo = limpiar_espacios(df_catalogo)
df_poblacion = limpiar_espacios(df_poblacion)
df_catalogo = normalizacion_columnas(df_catalogo)
df_poblacion = normalizacion_columnas(df_poblacion)

Anteriormente hacia el llamado a la api para generar los archivos, pero para efectos de compartir el proyecto, se usara la lectura directa

```
archivo_poblacion = ruta + poblacion
archivo_catalogo = ruta + catalogo_estados

poblacion_existe = os.path.exists(archivo_poblacion)
catalogo_existe = os.path.exists(archivo_catalogo)

if catalogo_existe:
  df_catalogo = pd.read_csv(archivo_catalogo, dtype={'clave_estado': str})
else:
  df_catalogo = generar_catalogo_areas_geograficas()

if poblacion_existe:
  df_poblacion = pd.read_csv(archivo_poblacion, dtype={'clave_estado': str})
else:
  df_poblacion = generar_archivo_poblacional(df_catalogo)


df_catalogo = limpiar_espacios(df_catalogo)
df_poblacion = limpiar_espacios(df_poblacion)
df_catalogo = normalizacion_columnas(df_catalogo)
df_poblacion = normalizacion_columnas(df_poblacion)
```



In [935]:
showDataFrame(df_catalogo)

df_catalogo

(32, 2)
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   clave_estado  32 non-null     object
 1   estado        32 non-null     object
dtypes: object(2)
memory usage: 644.0+ bytes
None
------------------------------
Registros duplicados
clave_estado    0
estado          0
dtype: int64


,clave_estado,estado
0,01,Aguascalientes
1,02,Baja California
2,03,Baja California Sur
3,04,Campeche
4,05,Coahuila de Zaragoza
5,06,Colima
6,07,Chiapas
7,08,Chihuahua
8,09,Ciudad de México
9,10,Durango


In [936]:
showDataFrame(df_poblacion)

df_poblacion

(479, 3)
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 479 entries, 0 to 478
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   clave_estado    479 non-null    object 
 1   anio_poblacion  479 non-null    int64  
 2   poblacion       479 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 11.4+ KB
None
------------------------------
Registros duplicados
clave_estado      0
anio_poblacion    0
poblacion         0
dtype: int64


,clave_estado,anio_poblacion,poblacion
0,01,1910,120511.0
1,01,1921,107581.0
2,01,1930,132900.0
3,01,1940,161693.0
4,01,1950,188075.0
...,...,...,...
474,32,2000,1353610.0
475,32,2005,1367692.0
476,32,2010,1490668.0
477,32,2015,1581575.0


# **Sección 2. Normalizando los dataset**

---




**Seccion 2.1 Limpieza de los estados**
---


In [937]:
nombres_oficiales = set(df_catalogo['estado'].unique())

**df_base_historica**

In [938]:
# Identificando los valores que no coinciden en df_base_historica
estados_base = set(df_base_historica['estado'].str.strip().unique())
no_coinciden = estados_base - nombres_oficiales

for estado in no_coinciden:
  print(estado)

Coahuila
CDMX, Estado de Mexico, Morelos, Puebla, Tlaxcala
CDMX, Morelos y Sonora
Oaxaca, Tamaulipas, Veracruz
Veracruz y Tamaulipas
Tamaulipas, Veracruz
Chihuahua, Durango, Estado de Mexico, Puebla, Sonora, Tlaxcala
Colima, Chiapas, Guerrero y Michoacán
Michoacán, Oaxaca, Sonora, Veracruz y Zacatecas
Todo el país
Coahuila, Chiapas, Distrito Federal, Hidalgo, Nayarit, Nuevo Leon, Oaxaca, Veracruz
Varios Estados
Michoacán
Coahuila, Chihuahua
Nayarit y Campeche
Veracruz
Chiapas, Michoacan, Nuevo Leon, Quintana Roo
Tabasco y Veracruz
Estado de México
Baja California Sur, Chichuahua y Michoacán


Construire diccionaarios para hacer una normalización de los estados que lo requieran, sepanrando los diccionarios en dos: uno para estados individualmente y uno para mixtos y variados

In [939]:
norm_map_base = {
    'Coahuila': 'Coahuila de Zaragoza',
    'Michoacán': 'Michoacán de Ocampo',
    'Veracruz': 'Veracruz de Ignacio de la Llave',
    'Estado de México': 'México',
}

multi_estado_o_pais = [
    'CDMX, Estado de Mexico, Morelos, Puebla, Tlaxcala',
    'Varios Estados',
    'Coahuila, Chihuahua',
    'Michoacán, Oaxaca, Sonora, Veracruz y Zacatecas',
    'Chihuahua, Durango, Estado de Mexico, Puebla, Sonora, Tlaxcala',
    'Tabasco y Veracruz',
    'Oaxaca, Tamaulipas, Veracruz',
    'Colima, Chiapas, Guerrero y Michoacán',
    'Nayarit y Campeche',
    'CDMX, Morelos y Sonora',
    'Baja California Sur, Chichuahua y Michoacán',
    'Chiapas, Michoacan, Nuevo Leon, Quintana Roo',
    'Veracruz y Tamaulipas',
    'Coahuila, Chiapas, Distrito Federal, Hidalgo, Nayarit, Nuevo Leon, Oaxaca, Veracruz',
    'Tamaulipas, Veracruz',
    'Todo el país',
]

In [940]:
# Aplicar el renombre
df_base_historica['estado'] = df_base_historica['estado'].str.strip().replace(norm_map_base)

# Marcar las filas multi-estado (no se eliminan, solo se etiquetan)
df_base_historica['estado_valido_para_join'] = ~df_base_historica['estado'].isin(multi_estado_o_pais)

In [941]:
# Verificación final
no_coinciden_final = set(df_base_historica['estado'].unique()) - nombres_oficiales
no_coinciden_lista = sorted(no_coinciden_final)
print("Después del renombre, valores que aún no coinciden (deberían ser solo los 16 multi-estado):")

printSeparador()
print(f"Total de elementos sin coincidencias {len(no_coinciden_final)}")
printSeparador()
for i, valor in enumerate(no_coinciden_lista, start=1):
    print(f'{i}.- {valor}')

Después del renombre, valores que aún no coinciden (deberían ser solo los 16 multi-estado):
------------------------------
Total de elementos sin coincidencias 16
------------------------------
1.- Baja California Sur, Chichuahua y Michoacán
2.- CDMX, Estado de Mexico, Morelos, Puebla, Tlaxcala
3.- CDMX, Morelos y Sonora
4.- Chiapas, Michoacan, Nuevo Leon, Quintana Roo
5.- Chihuahua, Durango, Estado de Mexico, Puebla, Sonora, Tlaxcala
6.- Coahuila, Chiapas, Distrito Federal, Hidalgo, Nayarit, Nuevo Leon, Oaxaca, Veracruz
7.- Coahuila, Chihuahua
8.- Colima, Chiapas, Guerrero y Michoacán
9.- Michoacán, Oaxaca, Sonora, Veracruz y Zacatecas
10.- Nayarit y Campeche
11.- Oaxaca, Tamaulipas, Veracruz
12.- Tabasco y Veracruz
13.- Tamaulipas, Veracruz
14.- Todo el país
15.- Varios Estados
16.- Veracruz y Tamaulipas


In [942]:
# Comprobar las veces que apareces estás multicoincidencias para determinar si no representan una cantidad significativa de información

filas_multi = df_base_historica[df_base_historica['estado'].isin(multi_estado_o_pais)]

print(f"Filas totales en df_base_historica: {len(df_base_historica)}")
print(f"Filas multi-estado/nacionales encontradas: {len(filas_multi)}")

printSeparador()

print("Desglose por valor")
print(filas_multi['estado'].value_counts())

Filas totales en df_base_historica: 10354
Filas multi-estado/nacionales encontradas: 20
------------------------------
Desglose por valor
estado
Varios Estados                                                                         5
CDMX, Estado de Mexico, Morelos, Puebla, Tlaxcala                                      1
Tamaulipas, Veracruz                                                                   1
Chiapas, Michoacan, Nuevo Leon, Quintana Roo                                           1
Chihuahua, Durango, Estado de Mexico, Puebla, Sonora, Tlaxcala                         1
Coahuila, Chihuahua                                                                    1
Oaxaca, Tamaulipas, Veracruz                                                           1
Coahuila, Chiapas, Distrito Federal, Hidalgo, Nayarit, Nuevo Leon, Oaxaca, Veracruz    1
Colima, Chiapas, Guerrero y Michoacán                                                  1
Michoacán, Oaxaca, Sonora, Veracruz y Zacatecas       

Debido a que representan solo 20 registros de un total de 10,354 (0.19%), el esfuerzo que requeriría tratar estos casos atípicos —cuyo campo `estado `corresponde a múltiples estados o cobertura nacional, sin forma de atribuirlos a un estado o región única— no está a la par del beneficio que aportarían al análisis. Por lo tanto, se descartaron por completo del dataset, quedando df_base_historica en 10,334 registros.

In [943]:
print(f"Filas antes de eliminar: {len(df_base_historica)}")

df_base_historica = df_base_historica[~df_base_historica['estado'].isin(multi_estado_o_pais)]
df_base_historica = df_base_historica.reset_index(drop=True)

print(f"Filas después de eliminar: {len(df_base_historica)}")

Filas antes de eliminar: 10354
Filas después de eliminar: 10334


---

**df_atlas**

In [944]:
# Identificando los valores que no coinciden en df_base_historica
estados_atlas = set(df_atlas['estado'].str.strip().unique())
no_coinciden_atlas = estados_atlas - nombres_oficiales

corregir_atlas = len(no_coinciden_atlas) > 0

if corregir_atlas > 0:
  print('Valores sin coincidir')
  for estado in no_coinciden_atlas:
    print(estado)
else:
  print('No hay valores sin coincidir')

Valores sin coincidir
Distrito Federal


El unico caso atípico que fue `Distrito Federal`, así que directamente remplaré esos casos por el nombre oficial y actual `Ciudad de México`

In [945]:
if corregir_atlas:
  df_atlas['estado'] = df_atlas['estado'].str.strip().replace({'Distrito Federal': 'Ciudad de México'})

In [946]:
no_coinciden_atlas_final = set(df_atlas['estado'].unique()) - nombres_oficiales
print(f"Valores sin coincidir: {len(no_coinciden_atlas_final)}")

Valores sin coincidir: 0


**Seccion 2.2 Formato de los campos de Fechas**
---


**df_base_historica**

In [947]:
df_base_historica.sample(5)

,fecha_de_inicio,fecha_de_fin,anio,clasificacion_del_fenomeno,tipo_de_fenomeno,estado,municipios_afectados,descripcion_general_de_los_danos,defunciones,poblacion_afectada,viviendas_danadas,escuelas,hospitales,comercios,area_de_cultivo_danada_/_pastizales_(h),total_de_danos_(millones_de_pesos),fuente,estado_valido_para_join
8227,15-may,15-may,2020,Sociorganizativo,Accidente de transporte,Tamaulipas,Reynosa,A las 18:15 horas se tuvo conocimiento de choq...,1,2,0,0,0,0,0,$0.57,CENACOM,True
6240,18-nov,19-nov,2015,Químico,Incendio urbano,México,Axapusco,Incendio en poliducto de 18 pulgadas ubicado e...,0,0,0,0,0,0,0,$0.12,CENACOM,True
3889,29-nov,29-nov,2010,Sociorganizativo,Accidente de transporte,Guanajuato,Victoria,"A las 07:00 horas, se registro la volcadura d...",2,4,0,0,0,0,0,$-,CENACOM,True
3754,13-dic,13-dic,2010,Químico,Incendio Urbano,Ciudad de México,Delegación Cuauhtémoc,"A las 10:30 horas, se registró un incendio en ...",0,170,0,0,0,0,0,$-,CENACOM,True
6533,1-ene,31-dic,2016,Químico,Incendio forestal,Morelos,Varios municipios,"En 2016 se reportaron 193 incendios, con 790.9...",0,0,0,0,0,0,1424.3,$1.42,CONAFOR,True


In [948]:
print(df_base_historica['fecha_de_inicio'].iloc[0])
print(type(df_base_historica['fecha_de_inicio'].iloc[0]))
print(df_base_historica['fecha_de_inicio'].dtype)

1-jun
<class 'str'>
object


Como las fechas como están ahora no me sirven, ya tecnicamente son texto simple, necesitare convertir la información actual a un dato que pueda ser considerado una fecha valida y luego ponerle el tipo de dato Fecha.


In [949]:
# Los meses están usando las primeras tres letras de sus nombres como no hay una conversión directa, tendré que hacer un diccionario para mapear está parte
meses_es = {
  'ene': 1,
  'feb': 2,
  'mar': 3,
  'abr': 4,
  'may': 5,
  'jun': 6,
  'jul': 7,
  'ago': 8,
  'sep': 9,
  'oct': 10,
  'nov': 11,
  'dic': 12,
}

In [950]:
def parsear_fecha_flexible(texto, anio_referencia, mes_minimo_para_cruce = None):
  if not isinstance(texto, str):
    return pd.NaT
  texto = texto.strip()
  if not texto:
    return pd.NaT

  # Limpiar typos comunes: doble slash "26//08/2022" -> "26/08/2022"
  texto_limpio = re.sub(r'/+', '/', texto)

  # CASO 1: formato con slashes "DD/MM/YYYY" o "DD/MM/YY" (ya trae año)
  m = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{2,4})$', texto_limpio)
  if m:
    dia, mes, anio_str = int(m.group(1)), int(m.group(2)), m.group(3)
    anio = int(anio_str) if len(anio_str) == 4 else 2000 + int(anio_str)
    try:
      return pd.Timestamp(year=anio, month=mes, day=dia)
    except (ValueError, TypeError):
      return pd.NaT

  # CASO 2: formato "D-mes" (ej. "18-dic") -- necesita anio_referencia
  m = re.match(r'^(\d{1,2})-([A-Za-z]{3})$', texto_limpio)
  if m:
    dia, mes_abrev = int(m.group(1)), m.group(2).lower()
    mes = meses_es.get(mes_abrev)
    if mes is None or pd.isna(anio_referencia):
      return pd.NaT
    anio = int(anio_referencia)

    # Si se está calculando fecha_de_fin y el mes es menor al de inicio, cruzó de año
    if mes_minimo_para_cruce is not None and mes < mes_minimo_para_cruce:
      anio += 1
    try:
      return pd.Timestamp(year=anio, month=mes, day=dia)
    except (ValueError, TypeError):
      return pd.NaT


  # CASO 3: no matchea ningún formato conocido (ej. "SD") -> NaT
  return pd.NaT

In [951]:
df_base_historica['fecha_de_inicio_dt'] = df_base_historica.apply(
    lambda row: parsear_fecha_flexible(row['fecha_de_inicio'], row['anio']),
    axis=1
)

In [952]:
df_base_historica['fecha_de_fin_dt'] = df_base_historica.apply(
    lambda row: parsear_fecha_flexible(
        row['fecha_de_fin'], row['anio'],
        mes_minimo_para_cruce=row['fecha_de_inicio_dt'].month if pd.notna(row['fecha_de_inicio_dt']) else None
    ),
    axis=1
)

In [953]:
mask_vacio_real = (
    df_base_historica['fecha_de_fin_dt'].isna() &
    df_base_historica['fecha_de_inicio_dt'].notna() &
    df_base_historica['fecha_de_fin'].isna()
)

df_base_historica.loc[mask_vacio_real, 'fecha_de_fin_dt'] = df_base_historica.loc[mask_vacio_real, 'fecha_de_inicio_dt']
print(f"Filas rellenadas con fecha_de_inicio (evento de 1 día asumido): {mask_vacio_real.sum()}")

Filas rellenadas con fecha_de_inicio (evento de 1 día asumido): 3


In [954]:
print("Fechas de inicio sin reconstruir:", df_base_historica['fecha_de_inicio_dt'].isna().sum())
print("Fechas de fin sin reconstruir:", df_base_historica['fecha_de_fin_dt'].isna().sum())

Fechas de inicio sin reconstruir: 2
Fechas de fin sin reconstruir: 3


In [955]:
print("Filas que siguen fallando en fecha_de_inicio_dt (irrecuperables):")
print(df_base_historica[df_base_historica['fecha_de_inicio_dt'].isna()][['fecha_de_inicio', 'fecha_de_fin', 'anio']])

Filas que siguen fallando en fecha_de_inicio_dt (irrecuperables):
   fecha_de_inicio fecha_de_fin  anio
45              SD       29-Sep  2001
52              SD           SD  2001


In [956]:
print("Filas que siguen fallando en fecha_de_fin_dt (irrecuperables):")
print(df_base_historica[df_base_historica['fecha_de_fin_dt'].isna()][['fecha_de_inicio', 'fecha_de_fin', 'anio']])

Filas que siguen fallando en fecha_de_fin_dt (irrecuperables):
      fecha_de_inicio fecha_de_fin  anio
29              1-dic          Año  2000
52                 SD           SD  2001
10204          11-jun  11/036/2024  2024


Como vemos hay declaratorias no pueden establecer un periodo de tiempo valido, por lo que para excluirlas usare una columna extra que almacena si dicha declaratoria tiene un periodo de tiempo correctamente establecido

In [957]:
df_base_historica['fecha_valida_completa'] = (
    df_base_historica['fecha_de_inicio_dt'].notna() &
    df_base_historica['fecha_de_fin_dt'].notna()
)

print(df_base_historica['fecha_valida_completa'].value_counts())

fecha_valida_completa
True     10330
False        4
Name: count, dtype: int64


In [958]:
df_base_historica

,fecha_de_inicio,fecha_de_fin,anio,clasificacion_del_fenomeno,tipo_de_fenomeno,estado,municipios_afectados,descripcion_general_de_los_danos,defunciones,poblacion_afectada,...,escuelas,hospitales,comercios,area_de_cultivo_danada_/_pastizales_(h),total_de_danos_(millones_de_pesos),fuente,estado_valido_para_join,fecha_de_inicio_dt,fecha_de_fin_dt,fecha_valida_completa
0,1-jun,1-jun,2000,Hidrometeorológico,Sequía,Aguascalientes,Varios Municipios,"Las sequias se forman con lentitud, afectan ma...",0,0,...,0,0,0,0,$25.10,CENAPRED,True,2000-06-01,2000-06-01,True
1,1-ene,31-dic,2000,Químico,Incendio forestal,Baja California,Varios Municipios,"En Mexico, estos fenomenos se presentan en la ...",0,0,...,0,0,0,0,$3.58,CENAPRED,True,2000-01-01,2000-12-31,True
2,16-sep,17-sep,2000,Hidrometeorológico,Ciclón tropical,Baja California Sur,"Los Cabos, La Paz","La Tormenta Tropical Miriam, afecto Severament...",0,0,...,0,0,0,0,$7.24,CENAPRED,True,2000-09-16,2000-09-17,True
3,30-sep,2-oct,2000,Hidrometeorológico,Ciclón tropical,Chiapas,Varios Municipios,El Huracan Keith se acerco peligrosamente a lo...,0,0,...,0,0,0,0,$25.56,CENAPRED,True,2000-09-30,2000-10-02,True
4,4-ene,4-ene,2000,Hidrometeorológico,Ciclón tropical,Chiapas,Varios Municipios,Reportado como otros,0,0,...,0,0,0,0,$38.15,CENAPRED,True,2000-01-04,2000-01-04,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10329,19-dic,19-dic,2024,Químico,Incendio urbano,Baja California Sur,Tijuana,"incendio de viviendas, en Bugambilia y De Las ...",2,72,...,0,0,0,-,$-,CENACOM,True,2024-12-19,2024-12-19,True
10330,22-dic,22-dic,2024,Sociorganizativo,Accidente de Transporte,Jalisco,Quitupan,Caída de una aeronave con matrícula XB - MLO e...,4,4,...,0,0,0,-,$0.20,CENACOM,True,2024-12-22,2024-12-22,True
10331,23-dic,23-dic,2024,Sociorganizativo,Accidente de Transporte,Guerrero,Técpan de Galeana,"Salida del camino de autobús de turismo, en la...",2,47,...,0,0,0,-,$0.09,CENACOM,True,2024-12-23,2024-12-23,True
10332,24-dic,24-dic,2024,Químico,Explosión,Jalisco,Atotonilco el Alto,Se registró la explosión de un polvorín clande...,1,4,...,0,0,0,-,$0.01,CENACOM,True,2024-12-24,2024-12-24,True


**df_atlas**

In [959]:
df_atlas.sample(5)

,identificador,estado,clave_estado,municipio,clave_municipio,tipo_declaratoria,clasificacion_fenomeno,tipo_fenomeno,fecha_publicacion,fecha_inicio,fecha_fin,observaciones
16503,19806,Chiapas,7,Mazatán,7054,Emergencia,Geológico,Sismo,16/07/2014,07/07/2014,07/07/2014,Presencia de sismo de 6.9 escala de Ritcher
10814,12011,Michoacán de Ocampo,16,Carácuaro,16013,Contingencia Climatológica,Hidrometeorológico,Sequía,10/12/2009,01/07/2009,31/08/2009,Sin Observaciones
31679,26858,Oaxaca,20,Calihualá,20011,Emergencia,Hidrometeorológico,Lluvias,11/10/2019,28/09/2019,29/09/2019,Lluvia severa e inundación fluvial y pluvial. ...
6930,8113,Oaxaca,20,San Jorge Nuchita,20164,Contingencia Climatológica,Hidrometeorológico,Sequía,28/12/2006,01/06/2006,30/09/2006,Sequia Atipica
20791,25960,Chiapas,7,Tapalapa,7090,Emergencia,Geológico,Sismo,18/09/2017,07/09/2017,07/09/2017,Declaratoria de emergencia extraordinaria sism...


`df_atlas` ya vienen en formato DD/MM/YYYY con año completo, y las 34,453 filas están no-nulas en las tres columnas de fecha. Así que se puede trabajar con una conversión más sencilla y directa que el `df_base_historica`

In [960]:
df_atlas['fecha_publicacion_dt'] = pd.to_datetime(df_atlas['fecha_publicacion'], dayfirst=True, errors='coerce')
df_atlas['fecha_inicio_dt'] = pd.to_datetime(df_atlas['fecha_inicio'], dayfirst=True, errors='coerce')
df_atlas['fecha_fin_dt'] = pd.to_datetime(df_atlas['fecha_fin'], dayfirst=True, errors='coerce')

In [961]:
print("Fechas sin convertir (deberían ser 0, o casi 0)")
print("fecha_publicacion:", df_atlas['fecha_publicacion_dt'].isna().sum())
print("fecha_inicio:", df_atlas['fecha_inicio_dt'].isna().sum())
print("fecha_fin:", df_atlas['fecha_fin_dt'].isna().sum())

Fechas sin convertir (deberían ser 0, o casi 0)
fecha_publicacion: 0
fecha_inicio: 0
fecha_fin: 0


**Seccion 2.3 Exclusion de fenomenos no requeridos**
---

In [962]:
def show_fenomenos_disponibles_base():
  combinaciones = df_base_historica[['tipo_de_fenomeno', 'clasificacion_del_fenomeno']].drop_duplicates()
  combinaciones = combinaciones.sort_values(['tipo_de_fenomeno', 'clasificacion_del_fenomeno'])

  tipos_unicos = df_base_historica['tipo_de_fenomeno'].unique()
  print(f"Total de tipos distintos: {len(tipos_unicos)}")
  print()

  for tipo in sorted(tipos_unicos):
    clasificaciones = combinaciones.loc[
        combinaciones['tipo_de_fenomeno'] == tipo, 'clasificacion_del_fenomeno'
    ].tolist()
    clasificaciones_texto = ', '.join(clasificaciones)
    print(f"{tipo:35s} -> {clasificaciones_texto}")

def show_clasificaciones_disponibles_base():
  tipos_unicos = df_base_historica['clasificacion_del_fenomeno'].unique()
  print(f"Total de clasificaciones distintos: {len(tipos_unicos)}")

  for tipo in sorted(tipos_unicos):
    print(tipo)

In [963]:
show_fenomenos_disponibles_base()

Total de tipos distintos: 64

Accidente                           -> Sociorganizativo
Accidente Aéreo                     -> Sociorganizativo
Accidente Ferroviario               -> Sociorganizativo
Accidente aéreo                     -> Sociorganizativo
Accidente de Transporte             -> Sociorganizativo
Accidente de trabajo                -> Sociorganizativo
Accidente de transporte             -> Químico, Sociorganizativo
Accidente marítimo                  -> Sociorganizativo
Accidente urbano                    -> Sociorganizativo
Accidentre de Trabajo               -> Sociorganizativo
Actividad volcánica                 -> Geológico
Acto de sabotaje                    -> Sociorganizativo
Agrietamiento                       -> Geológico
Ahogamiento                         -> Sociorganizativo
Amenaza de bomba                    -> Sociorganizativo
Bacteria                            -> Sanitario
Bajas Temperaturas                  -> Hidrometeorológico
Bajas temperaturas          

In [964]:
show_clasificaciones_disponibles_base()

Total de clasificaciones distintos: 5
Geológico
Hidrometeorológico
Químico
Sanitario
Sociorganizativo


Como veo que hay algunos declaratorias que son del mismo tipo pero no los considera así por una variación entre mayusculas y minusculas, usare una función para normalizar la letra capitual en la columna `tipo_de_fenomeno`.

Hay un caso especial con `Lluvia` y `Lluvias` que son la mismo tipo, pero está escritas distinta manera, así que todos los registros de  `Lluvia` pararan a ser `Lluvias`.

En el caso de `clasificacion_del_fenomeno` no hay necesidad de hacer algo más, la información no muestra incosistenncias o valores repetidos.

In [965]:
df_base_historica = normalizar_capitalizacion(
    df_base_historica,
    columnas=['tipo_de_fenomeno', 'clasificacion_del_fenomeno']
)

df_base_historica['tipo_de_fenomeno'] = df_base_historica['tipo_de_fenomeno'].replace({'Lluvia': 'Lluvias'})

In [966]:
show_fenomenos_disponibles_base()

Total de tipos distintos: 58

Accidente                           -> Sociorganizativo
Accidente aéreo                     -> Sociorganizativo
Accidente de trabajo                -> Sociorganizativo
Accidente de transporte             -> Químico, Sociorganizativo
Accidente ferroviario               -> Sociorganizativo
Accidente marítimo                  -> Sociorganizativo
Accidente urbano                    -> Sociorganizativo
Accidentre de trabajo               -> Sociorganizativo
Actividad volcánica                 -> Geológico
Acto de sabotaje                    -> Sociorganizativo
Agrietamiento                       -> Geológico
Ahogamiento                         -> Sociorganizativo
Amenaza de bomba                    -> Sociorganizativo
Bacteria                            -> Sanitario
Bajas temperaturas                  -> Hidrometeorológico
Bloqueo                             -> Sociorganizativo
Caída de barda                      -> Sociorganizativo
Ciclón tropical             

**Filtrado de fenomenos**

Como se planteo al inicio de este documento, se planen analizar desastres naturales. Por lo tanto, excluire los fenómenos creados por el hombre.

Se incluyeron:
* Geológico: Incluido, excepto "Colapso de estructura"
* Hidrometeorológico
* Químico: Solo se incluye Incendio forestal, el resto se excluyeron

Se excluyeron:
* Químico: Explosiones, fugas, derrames, incendios urbanos — industriales
* Sanitario: Por completo
* Sociorganizativo: Por completo

In [967]:
condicion_natural = (
    (df_base_historica['clasificacion_del_fenomeno'].isin(['Geológico', 'Hidrometeorológico']) &
     (df_base_historica['tipo_de_fenomeno'] != 'Colapso de estructura')) |
    ((df_base_historica['clasificacion_del_fenomeno'] == 'Químico') &
     (df_base_historica['tipo_de_fenomeno'] == 'Incendio forestal'))
)

In [968]:
print(f"Filas antes: {len(df_base_historica)}")
df_base_historica = df_base_historica[condicion_natural].reset_index(drop=True)
print(f"Filas después (desastres naturales): {len(df_base_historica)}")

Filas antes: 10334
Filas después (desastres naturales): 4957


**Seccion 2.4 Formato numerico de los campos cuantitativos**
---


In [969]:
def limpiar_numero(texto):
  if not isinstance(texto, str):
        return texto

  texto = texto.strip()
  if texto in ('$-', '$ -', '-'):
    return '0'

  return re.sub(r'[\$,]', '', texto).strip()

In [970]:
columnas_numericas = [
    'defunciones',
    'poblacion_afectada',
    'viviendas_danadas',
    'escuelas',
    'hospitales',
    'comercios',
    'area_de_cultivo_danada_/_pastizales_(h)',
    'total_de_danos_(millones_de_pesos)'
]

for col in columnas_numericas:
  df_base_historica[col] = df_base_historica[col].apply(limpiar_numero)
  df_base_historica[col] = pd.to_numeric(df_base_historica[col], errors='coerce')

# Verificación: cuántos NaN quedaron por columna (valores tipo "SD", "sd", "NSR")
print(df_base_historica[columnas_numericas].isna().sum())

defunciones                                 8
poblacion_afectada                         45
viviendas_danadas                          47
escuelas                                   56
hospitales                                 55
comercios                                  27
area_de_cultivo_danada_/_pastizales_(h)    80
total_de_danos_(millones_de_pesos)          9
dtype: int64


In [971]:
print("Tipos de dato después de la conversión:")
print(df_base_historica[columnas_numericas].dtypes)
printSeparador()
print("Nulos por columna (valores 'sin dato' genuinos tipo SD/NSR):")
print(df_base_historica[columnas_numericas].isna().sum())
printSeparador()
print(f"Columnas restantes en el dataframe: {df_base_historica.shape[1]}")

Tipos de dato después de la conversión:
defunciones                                float64
poblacion_afectada                         float64
viviendas_danadas                          float64
escuelas                                   float64
hospitales                                 float64
comercios                                  float64
area_de_cultivo_danada_/_pastizales_(h)    float64
total_de_danos_(millones_de_pesos)         float64
dtype: object
------------------------------
Nulos por columna (valores 'sin dato' genuinos tipo SD/NSR):
defunciones                                 8
poblacion_afectada                         45
viviendas_danadas                          47
escuelas                                   56
hospitales                                 55
comercios                                  27
area_de_cultivo_danada_/_pastizales_(h)    80
total_de_danos_(millones_de_pesos)          9
dtype: int64
------------------------------
Columnas restantes en el dataframe

In [972]:
df_base_historica = df_base_historica.drop(columns=['fuente'])

# **Sección 3. Unificacion de los dataset**

---




**Seccion 3.1 Agregar `df_atlas` a nivel evento (Estado + Clasificación + ventana de fechas)**
---

In [973]:
# Es un periodo de gracia para hacer el join entre los datasets
tolerancia = pd.Timedelta(days=10)

In [974]:
for columna in df_atlas.columns.tolist():
  print(columna)

identificador
estado
clave_estado
municipio
clave_municipio
tipo_declaratoria
clasificacion_fenomeno
tipo_fenomeno
fecha_publicacion
fecha_inicio
fecha_fin
observaciones
fecha_publicacion_dt
fecha_inicio_dt
fecha_fin_dt


In [975]:
for columna in df_base_historica.columns.tolist():
  print(columna)

fecha_de_inicio
fecha_de_fin
anio
clasificacion_del_fenomeno
tipo_de_fenomeno
estado
municipios_afectados
descripcion_general_de_los_danos
defunciones
poblacion_afectada
viviendas_danadas
escuelas
hospitales
comercios
area_de_cultivo_danada_/_pastizales_(h)
total_de_danos_(millones_de_pesos)
estado_valido_para_join
fecha_de_inicio_dt
fecha_de_fin_dt
fecha_valida_completa


In [976]:
atlas_por_estado = {estado: grupo for estado, grupo in df_atlas.groupby('estado')}

In [977]:
atlas_por_estado

{'Aguascalientes':        identificador          estado  clave_estado                  municipio  \
 1544            2686  Aguascalientes             1             Aguascalientes   
 2521            3663  Aguascalientes             1         San José de Gracia   
 2522            3664  Aguascalientes             1            Rincón de Romos   
 2523            3665  Aguascalientes             1                   Calvillo   
 2524            3666  Aguascalientes             1                   Asientos   
 ...              ...             ...           ...                        ...   
 28974          26570  Aguascalientes             1        Pabellón de Arteaga   
 28975          26570  Aguascalientes             1  San Francisco de los Romo   
 28976          26570  Aguascalientes             1         San José de Gracia   
 28977          26570  Aguascalientes             1                   Tepezalá   
 30112          26697  Aguascalientes             1                   Calvillo  

In [978]:
def show_estodo_agrupado(key):
  if key not in atlas_por_estado:
    print(f"No existe el estado {key}")
    return

  print(f"Estado {key}")
  printSeparador()
  grupo = atlas_por_estado[key]
  return grupo

show_estodo_agrupado('Aguascalientes')


Estado Aguascalientes
------------------------------


,identificador,estado,clave_estado,municipio,clave_municipio,tipo_declaratoria,clasificacion_fenomeno,tipo_fenomeno,fecha_publicacion,fecha_inicio,fecha_fin,observaciones,fecha_publicacion_dt,fecha_inicio_dt,fecha_fin_dt
1544,2686,Aguascalientes,1,Aguascalientes,1001,Emergencia,Hidrometeorológico,Lluvias,14/11/2003,07/09/2003,09/09/2003,Lluvias Atipicas,2003-11-14,2003-09-07,2003-09-09
2521,3663,Aguascalientes,1,San José de Gracia,1008,Contingencia Climatológica,Hidrometeorológico,"Nevadas, Heladas, Granizadas",24/08/2004,28/07/2004,29/07/2004,Granizada,2004-08-24,2004-07-28,2004-07-29
2522,3664,Aguascalientes,1,Rincón de Romos,1007,Contingencia Climatológica,Hidrometeorológico,"Nevadas, Heladas, Granizadas",24/08/2004,28/07/2004,29/07/2004,Granizada,2004-08-24,2004-07-28,2004-07-29
2523,3665,Aguascalientes,1,Calvillo,1003,Contingencia Climatológica,Hidrometeorológico,"Nevadas, Heladas, Granizadas",24/08/2004,28/07/2004,29/07/2004,Granizada,2004-08-24,2004-07-28,2004-07-29
2524,3666,Aguascalientes,1,Asientos,1002,Contingencia Climatológica,Hidrometeorológico,"Nevadas, Heladas, Granizadas",24/08/2004,28/07/2004,29/07/2004,Granizada,2004-08-24,2004-07-28,2004-07-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28974,26570,Aguascalientes,1,Pabellón de Arteaga,1006,Contingencia Climatológica,Hidrometeorológico,Sequía,23/11/2011,01/06/2011,31/07/2011,Sequía atípica,2011-11-23,2011-06-01,2011-07-31
28975,26570,Aguascalientes,1,San Francisco de los Romo,1011,Contingencia Climatológica,Hidrometeorológico,Sequía,23/11/2011,01/06/2011,31/07/2011,Sequía atípica,2011-11-23,2011-06-01,2011-07-31
28976,26570,Aguascalientes,1,San José de Gracia,1008,Contingencia Climatológica,Hidrometeorológico,Sequía,23/11/2011,01/06/2011,31/07/2011,Sequía atípica,2011-11-23,2011-06-01,2011-07-31
28977,26570,Aguascalientes,1,Tepezalá,1009,Contingencia Climatológica,Hidrometeorológico,Sequía,23/11/2011,01/06/2011,31/07/2011,Sequía atípica,2011-11-23,2011-06-01,2011-07-31


In [979]:
def get_declaratoria(oficial = False, n_municipios = 0, tipo = None):
  return {
      'tuvo_declaratoria_oficial': 'Sí' if oficial else 'No',
      'n_municipios_declarados': n_municipios,
      'tipos_declaratoria': tipo,
  }

def agregar_declaratorias(row):
  grupo = atlas_por_estado.get(row['estado'])
  if grupo is None:
    return pd.Series(get_declaratoria())

  # Filtrado por misma clasificacion de fenómeno
  grupo_filtrado = grupo[grupo['clasificacion_fenomeno'] == row['clasificacion_del_fenomeno']]

  if grupo_filtrado.empty:
    return pd.Series(get_declaratoria())

  # Filtrar por ventana de fechas con la tolerancia ya definida
  fi_tol = row['fecha_de_inicio_dt'] - tolerancia
  ff_tol = row['fecha_de_fin_dt'] + tolerancia

  coincidencias = grupo_filtrado[
      (grupo_filtrado['fecha_inicio_dt'] <= ff_tol) &
      (grupo_filtrado['fecha_fin_dt'] >= fi_tol)
  ]

  if coincidencias.empty:
    return pd.Series(get_declaratoria())

  return pd.Series(get_declaratoria(
      True,
      coincidencias['municipio'].nunique(),
      ', '.join(sorted(coincidencias['tipo_declaratoria'].unique()))
  ))

In [980]:
n_filas_antes = len(df_base_historica)

nuevas_columnas = df_base_historica.apply(agregar_declaratorias, axis=1)
df_base_historica = pd.concat([df_base_historica, nuevas_columnas], axis=1)

In [981]:
n_filas_despues = len(df_base_historica)
assert n_filas_antes == n_filas_despues, "¡Se perdieron o duplicaron filas en el join!"

print(f"Filas antes del join: {n_filas_antes}")
print(f"Filas después del join: {n_filas_despues}")
print("✅ Integridad confirmada: el número de filas no cambió")
print()
print(df_base_historica['tuvo_declaratoria_oficial'].value_counts())
print()
print(f"Porcentaje con declaratoria oficial encontrada: "
      f"{100 * (df_base_historica['tuvo_declaratoria_oficial']=='Sí').mean():.1f}%")

Filas antes del join: 4957
Filas después del join: 4957
✅ Integridad confirmada: el número de filas no cambió

tuvo_declaratoria_oficial
No    2871
Sí    2086
Name: count, dtype: int64

Porcentaje con declaratoria oficial encontrada: 42.1%


**Seccion 3.2 Agregar estados y la tasa poblacional a df_problacion**
---

In [982]:
showDataFrame(df_base_historica)

(4957, 23)
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4957 entries, 0 to 4956
Data columns (total 23 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   fecha_de_inicio                          4957 non-null   object        
 1   fecha_de_fin                             4955 non-null   object        
 2   anio                                     4957 non-null   int64         
 3   clasificacion_del_fenomeno               4957 non-null   object        
 4   tipo_de_fenomeno                         4957 non-null   object        
 5   estado                                   4957 non-null   object        
 6   municipios_afectados                     4928 non-null   object        
 7   descripcion_general_de_los_danos         4956 non-null   object        
 8   defunciones                              4949 non-null   float64       
 9  

In [983]:
showDataFrame(df_poblacion)

(479, 3)
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 479 entries, 0 to 478
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   clave_estado    479 non-null    object 
 1   anio_poblacion  479 non-null    int64  
 2   poblacion       479 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 11.4+ KB
None
------------------------------
Registros duplicados
clave_estado      0
anio_poblacion    0
poblacion         0
dtype: int64


In [984]:
df_poblacion = df_poblacion.merge(df_catalogo, on='clave_estado', how='left')

estados_sin_nombre = df_poblacion['estado'].isna().sum()
if estados_sin_nombre > 0:
  print(f'Cantidad de estados sin nombre: {estados_sin_nombre}')
  for estado in df_poblacion[df_poblacion['estado'].isna()]['estado']:
    print(estado)
else:
  print('Todos los estados tienen un nombre oficial')

Todos los estados tienen un nombre oficial


In [985]:
df_poblacion_sorted = df_poblacion.sort_values('anio_poblacion').reset_index(drop=True)
df_base_sorted = df_base_historica.sort_values('anio').reset_index(drop=True)

In [986]:
n_filas_antes = len(df_base_sorted)

df_base_historica = pd.merge_asof(
    df_base_sorted,
    df_poblacion_sorted[['estado', 'anio_poblacion', 'poblacion']],
    left_on='anio',
    right_on='anio_poblacion',
    by='estado',
    direction='nearest'
)

In [987]:
n_filas_despues = len(df_base_historica)
assert n_filas_antes == n_filas_despues, "¡Se perdieron o duplicaron filas!"
print(f"Filas antes: {n_filas_antes}")
print(f"Filas después: {n_filas_despues}")

Filas antes: 4957
Filas después: 4957


In [988]:
df_base_historica['pct_poblacion_afectada'] = (
    df_base_historica['poblacion_afectada'] / df_base_historica['poblacion'] * 100
)

print(df_base_historica[['estado', 'anio', 'anio_poblacion', 'poblacion',
                          'poblacion_afectada', 'pct_poblacion_afectada']].sample(5))

                               estado  anio  anio_poblacion  poblacion  \
981                   San Luis Potosí  2006            2005  2410414.0   
3713              Michoacán de Ocampo  2018            2020  4748846.0   
4736  Veracruz de Ignacio de la Llave  2023            2020  8062579.0   
876                        Guanajuato  2005            2005  4893812.0   
2469                        Chihuahua  2012            2010  3406465.0   

      poblacion_afectada  pct_poblacion_afectada  
981                 10.0                0.000415  
3713                 1.0                0.000021  
4736               174.0                0.002158  
876                  0.0                0.000000  
2469              1888.0                0.055424  


# **Sección 4. Resolucion de preguntas**

---




**Seccion 4.1 Verdaderamente, ¿en mi natal estado de Sonora ha empeorado la sequía a lo largo de los últimos 20 años? De ser así, responder a la pregunta, ¿Cuánto ha empeorado la sequía en mi comunidad a lo largo de los años? O en caso contrario, ¿cuanto a disminuido está problematica?**
---

In [989]:
sonora_sequia = df_base_historica[
    (df_base_historica['estado'] == 'Sonora') &
    (df_base_historica['tipo_de_fenomeno'] == 'Sequía')
].copy()

print(f"Eventos de sequía en Sonora: {len(sonora_sequia)}")

Eventos de sequía en Sonora: 7


In [990]:
serie_danos = sonora_sequia.groupby('anio')['total_de_danos_(millones_de_pesos)'].sum()
serie_danos_completa = serie_danos.reindex(range(2000, 2025), fill_value=0)

serie_frecuencia = sonora_sequia.groupby('anio').size()
serie_frecuencia_completa = serie_frecuencia.reindex(range(2000, 2025), fill_value=0)

In [991]:
print("\n--- Tendencia en DAÑOS ECONÓMICOS por sequía ---")
resultado_danos = mk.original_test(serie_danos_completa.values)
print(resultado_danos)

print("\n--- Tendencia en FRECUENCIA de eventos de sequía ---")
resultado_frecuencia = mk.original_test(serie_frecuencia_completa.values)
print(resultado_frecuencia)


--- Tendencia en DAÑOS ECONÓMICOS por sequía ---
Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.7620293552427218), z=np.float64(-0.3028169632097039), Tau=np.float64(-0.03333333333333333), s=np.float64(-10.0), var_s=np.float64(883.3333333333334), slope=np.float64(0.0), intercept=np.float64(0.0))

--- Tendencia en FRECUENCIA de eventos de sequía ---
Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.31149141455567797), z=np.float64(-1.0120975482378414), Tau=np.float64(-0.11), s=np.float64(-33.0), var_s=np.float64(999.6666666666666), slope=np.float64(0.0), intercept=np.float64(0.0))


**Conclusion**

Con solo 7 eventos de sequía registrados en 25 años (2000-2024), la prueba de Mann-Kendall no encuentra una tendencia estadísticamente significativa, ni en daños económicos (p=0.762) ni en frecuencia de eventos (p=0.311). Esto significa que, con la evidencia disponible en la base de datos oficial de desastres, no se puede afirmar con rigor estadístico que la sequía haya empeorado de forma sostenida en el periodo analizado.

Esto no contradice necesariamente la percepción de que la sequía actual es severa — el episodio más reciente (2019-2020) sí fue el más dañino económicamente ($482M de pesos) y el de mayor cobertura municipal según declaratorias (46 municipios en 2020 vs. 11 en 2011). Pero la muestra es demasiado pequeña y dispersa para distinguir eso de una fluctuación normal. La sequía, al ser un fenómeno de evolución lenta, tiende a estar subrepresentada en bases de datos de "eventos" puntuales como esta.

**Seccion 4.2 ¿En qué estados hay más desastres naturales y cuales son el tipo más frecuente y en que grado?**
---

In [992]:
conteo = df_base_historica.groupby('estado').size().sort_values(ascending=False)
top10 = conteo.head(10).index

In [993]:
resumen_top10 = []
for estado in top10:
  sub = df_base_historica[df_base_historica['estado'] == estado]
  resumen_top10.append({
      'estado': estado,
      'n_eventos': len(sub),
      'tipo_mas_frecuente': sub['tipo_de_fenomeno'].value_counts().idxmax(),
      'danos_totales': sub['total_de_danos_(millones_de_pesos)'].sum(),
      'defunciones_totales_estado': sub['defunciones'].sum(),
  })

resumen_top10 = pd.DataFrame(resumen_top10)

In [994]:
resumen_top10

,estado,n_eventos,tipo_mas_frecuente,danos_totales,defunciones_totales_estado
0,Veracruz de Ignacio de la Llave,423,Lluvias,80080.81,354.0
1,Chiapas,370,Lluvias,57371.15,292.0
2,Oaxaca,264,Lluvias,45772.89,414.0
3,Guerrero,251,Lluvias,132788.51,329.0
4,Guanajuato,225,Lluvias,3529.10,52.0
5,Michoacán de Ocampo,212,Lluvias,9081.24,104.0
6,Jalisco,207,Lluvias,11431.61,85.0
7,Puebla,197,Lluvias,12887.86,231.0
8,Sonora,184,Lluvias,8406.66,496.0
9,México,182,Lluvias,6825.95,123.0


**Conclusión**

Es posible observar que efectivamente, el top 10 de desastres es encabezado por estados del sur y centro del país, donde se ve un que los daños economicos son sumamente alto. Sin embargo, si nos fijamos en la columna de defunciones, se puede ver que los valores varian mucho y dependiendo del lugar puede ser más o menor la perdida de vidas humanas.

Curiosamente, estoy viendo que Sonora terminó colándose en el ranking, y sorprendentemente siendo lluvias el fenómeno más visto en dicho estado. Y si bien sus eventos no reportan la peor pérdida económica, sí tiene el valor más alto de defunciones del ranking, con un total de 496 defunciones registradas entre todos sus eventos de desastre natural.

**Seccion 4.3 ¿Cuáles son los mayores desastres naturales a nivel región?**
---

In [995]:
region_por_estado = {estado: region for region, lista in estados_por_region.items() for estado in lista}
df_base_historica['region'] = df_base_historica['estado'].map(region_por_estado)

resumen_region = df_base_historica.groupby('region').agg(
    n_eventos=('estado', 'count'),
    danos_totales=('total_de_danos_(millones_de_pesos)', 'sum'),
    defunciones_totales=('defunciones', 'sum'),
).sort_values('danos_totales', ascending=False)


In [996]:
resumen_region

,n_eventos,danos_totales,defunciones_totales
region,,,
Sur,885,235932.55,1035.0
Sureste,751,194039.52,537.0
Norte,1233,133678.25,2150.0
Centro,943,86147.59,865.0
Occidente,1145,53792.18,481.0


In [997]:
for region in resumen_region.index:
    sub = df_base_historica[df_base_historica['region'] == region]
    fila = sub.loc[sub['total_de_danos_(millones_de_pesos)'].idxmax()]
    print(f"{region}: {fila['estado']} - {fila['tipo_de_fenomeno']} ({fila['anio']}) - ${fila['total_de_danos_(millones_de_pesos)']:,.1f} M")

Sur: Guerrero - Ciclón tropical (2023) - $84,207.0 M
Sureste: Tabasco - Lluvias (2007) - $31,871.3 M
Norte: Baja California Sur - Ciclón tropical (2014) - $24,133.2 M
Centro: Ciudad de México - Sismo (2017) - $43,996.1 M
Occidente: Zacatecas - Sequía (2010) - $2,770.0 M


**Conclusiones**

Estos cinco eventos al buscarlos encontre especificamente los nombres para cada una de esas tragedias:

En el sur el Huracán Otis es uno de los peores desastres que hemos visto en está declada, y por lo analizado tuvo una perdida de $84,207.0 M, sin contar la perdida de vidas humanas.

En el sureste, las lluvias que afectaron a los estados de tabasco y chiapas provocando inundaciones en la región, con una perdida de $31,871.3 M de pesos.

En el norte en Baja California Sur, soportando el desastre provocado por el Huracán Odile, dejo una destrucción valuada en $24,133.2 M de pesos.

Aunque sin duda, probablemente el caso más mediatico y conocido, fue el terromoto de CDMX del 2017, donde se sufrieron perdidas economicas de aproximadamente $43,996.1 M.

El caso que probablemente paso en su momento más desapercibido fue la sequía en la región Occidente, afectando al estado de Zacatecas, con un costo ecónomico de $2,770.0 M de pesos.

**Seccion 4.4 ¿Qué incidentes con muchas defunciones o daños altos no tuvieron declaratoria oficial y/o contaron con una información temporal mal definida?**
---

In [998]:
umbral_defunciones = df_base_historica['defunciones'].quantile(0.95)
umbral_danos = df_base_historica['total_de_danos_(millones_de_pesos)'].quantile(0.95)

In [999]:
alto_impacto = df_base_historica[
    (df_base_historica['defunciones'] >= umbral_defunciones) |
    (df_base_historica['total_de_danos_(millones_de_pesos)'] >= umbral_danos)
]

In [1000]:
problema = alto_impacto[
    (alto_impacto['tuvo_declaratoria_oficial'] == 'No') |
    (~alto_impacto['fecha_valida_completa'])
]

In [1001]:

print(f"Eventos de alto impacto: {len(alto_impacto)}")
print(f"Sin declaratoria y/o fecha mal definida: {len(problema)}")

printSeparador()

print(problema.sort_values('defunciones', ascending=False)[
    ['estado','tipo_de_fenomeno','anio','defunciones','total_de_danos_(millones_de_pesos)']
].head(20))

Eventos de alto impacto: 484
Sin declaratoria y/o fecha mal definida: 116
------------------------------
                   estado     tipo_de_fenomeno  anio  defunciones  \
4723               Sonora  Temperatura extrema  2023        118.0   
4756           Nuevo León  Temperatura extrema  2023        103.0   
4837               Sonora  Temperatura extrema  2024         48.0   
4840      Baja California  Temperatura extrema  2024         47.0   
1321               Puebla        Deslizamiento  2007         32.0   
4748           Tamaulipas  Temperatura extrema  2023         32.0   
4834              Tabasco  Temperatura extrema  2024         24.0   
47                 Oaxaca      Ciclón tropical  2001         23.0   
4838           Tamaulipas  Temperatura extrema  2024         21.0   
4568            Chihuahua   Bajas temperaturas  2022         21.0   
2025              Chiapas        Deslizamiento  2010         19.0   
4178      Baja California  Temperatura extrema  2021         19.0  

In [1002]:
tasa_reconocimiento = alto_impacto.groupby('tipo_de_fenomeno')['tuvo_declaratoria_oficial'].apply(
    lambda x: (x == 'Sí').mean() * 100
).sort_values(ascending=False)

In [1003]:
tasa_reconocimiento

,tuvo_declaratoria_oficial
tipo_de_fenomeno,
Helada,100.000000
Tormenta eléctrica,100.000000
Tormenta tropical,100.000000
Sequía,94.736842
Ciclón tropical,94.690265
Sismo,94.444444
Inundación,88.235294
Tormenta severa,80.000000
Bajas temperaturas,79.166667


**Conclusiones**

Los desastres naturales de tipo hidrometeorológicos como ciclones, tormentas e inundaciones, casi siempre son reportado y generan una declaratoria oficial documentada. Mientras, los fenómenos geológicos e incendios forestales que generalmente toman más tiempo para evolucionar a un problema de interes son menos documentados.

**Seccion 4.5 ¿Cuáles son los eventos que tienen un mayor impacto económico y con qué frecuencia pasan?**
---

In [1004]:
resumen_tipo = df_base_historica.groupby('tipo_de_fenomeno').agg(
    n_eventos=('tipo_de_fenomeno', 'count'),
    danos_totales=('total_de_danos_(millones_de_pesos)', 'sum'),
    danos_promedio_por_evento=('total_de_danos_(millones_de_pesos)', 'mean'),
)

In [1005]:
resumen_tipo.sort_values('danos_totales', ascending=False)

,n_eventos,danos_totales,danos_promedio_por_evento
tipo_de_fenomeno,,,
Ciclón tropical,232,365556.06,1575.672672
Lluvias,2111,154431.13,73.398826
Sismo,58,101647.07,1752.535690
Inundación,161,26008.71,161.544783
Sequía,155,23824.22,153.704645
Incendio forestal,839,17623.94,21.005888
Bajas temperaturas,359,3906.00,10.880223
Deslizamiento,110,2669.56,24.268727
Proceso de remoción en masa,97,2571.67,26.512062


In [1006]:
resumen_tipo.sort_values('n_eventos', ascending=False)

,n_eventos,danos_totales,danos_promedio_por_evento
tipo_de_fenomeno,,,
Lluvias,2111,154431.13,73.398826
Incendio forestal,839,17623.94,21.005888
Bajas temperaturas,359,3906.00,10.880223
Temperatura extrema,322,348.81,1.083261
Ciclón tropical,232,365556.06,1575.672672
Tormenta severa,171,2077.32,12.148070
Inundación,161,26008.71,161.544783
Sequía,155,23824.22,153.704645
Fuertes vientos,129,406.30,3.174219


In [1007]:
resumen_tipo.sort_values('danos_promedio_por_evento', ascending=False)

,n_eventos,danos_totales,danos_promedio_por_evento
tipo_de_fenomeno,,,
Sismo,58,101647.07,1752.535690
Ciclón tropical,232,365556.06,1575.672672
Inundación,161,26008.71,161.544783
Sequía,155,23824.22,153.704645
Tornado,1,125.17,125.170000
Tormenta tropical,10,1096.73,109.673000
Lluvias,2111,154431.13,73.398826
Otros,1,55.51,55.510000
Proceso de remoción en masa,97,2571.67,26.512062


**Conclusiones**

Está pregunta es dependiente de la métrica que se use, en este caso mostre tres: `danos_promedio_por_evento`, `danos_totales` y `n_eventos`, donde cada uno mostro tres distintos escenarios de cuales son sus eventos más representativos y que generan un mayor impacto.

Sin embargo, analizando los datos y los resultados, se puede ver que `Ciclón tropical` se encuentra en el top 3 tanto de `danos_promedio_por_evento` como de `danos_totales`, pero no en el de  `n_eventos`. Está situación también sucecde en el caso de `sismo`.

Por otro lado, `lluvias` si es incluido en el top 3 de `n_eventos` y de `danos_totales`, siendo considerado entonces como uno de los eventos más baratos.

Asimismo, aunque `incendio forestal` y `bajas temperaturas` ocurren frecuentemente, paracen no tener un impacto significativo en cuestión economica, pues no aparecen  en `danos_promedio_por_evento` y `danos_totales` como parte del top.

# **Bibliografía**

---

> Echeverría, M. (2026, 8 de abril). Presa Oviáchic en Valle del Yaqui sigue baja con apenas 16.7% de almacenamiento y preocupa a productores del campo. El Imparcial. https://www.elimparcial.com/son/ciudad-obregon/2026/04/08/presa-oviachic-en-valle-del-yaqui-sigue-baja-con-apenas-167-de-almacenamiento-y-preocupa-a-productores-del-campo/


> De la Hoya, A. (2026, 18 de mayo). Sequía en Sonora: Ocho municipios sufren sequía y 16 están anormalmente secos según Conagua. Tribuna de San Luis. https://oem.com.mx/tribunadesanluis/local/sequia-en-sonora-ocho-municipios-sufren-sequia-y-16-estan-anormalmente-secos-segun-conagua-30065858


> Durán, D. (2026, 26 de marzo). Sequía extrema castiga al norte de México. El Independiente. https://elindependiente.mx/estados/2026/03/26/sequia-extrema-castiga-al-norte-de-mexico/


> Centro Nacional de Prevención de Desastres (CENAPRED). (2024). Base de datos sobre el impacto socioeconómico de los daños y pérdidas ocasionados por los desastres en México de 2000 a 2024 [Conjunto de datos]. Secretaría de Gobernación. http://www.atlasnacionalderiesgos.gob.mx/archivo/descargas.html

